# Install Libraries

In [0]:
!pip install openai pypdf databricks-vectorsearch tavily-python brevo-python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# Import Packages

In [0]:
%restart_python

In [0]:
from pypdf import PdfReader
from agents import Agent, Runner, trace, set_default_openai_client, OpenAIChatCompletionsModel, function_tool
from openai import OpenAI
from tavily import TavilyClient
import brevo_python
from brevo_python.rest import ApiException
from typing import Dict
mlflow.openai.autolog()

# PRE-REQUISITE STARTS HERE

# Read PDF

In [0]:
file_path = "/Volumes/workspace/llm_dev/landing/docs/Sayon_Bhattacharjee_CV.pdf"
reader = PdfReader(file_path)
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text
name = "Sayon Bhattacharjee"  
# print(linkedin)        

# Custom Text Splitter

In [0]:
def my_text_splitter(text, chunk_size, overlap):
    chunks=[]
    start=0
    while start<=len(text):
        end = start+chunk_size
        print('Start:',start)
        print('End:',end)
        chunk = text[start:end]
        print('chunk:',chunk)
        chunks.append(chunk)
        print('Chunk array:',chunks)
        start = end-overlap
        print('Start modified to:',start)
    print('Final Chunk:',chunks)        
    return chunks

In [0]:
my_text_splitter(text=linkedin, chunk_size=1000, overlap=100)
chunks = my_text_splitter(text=linkedin, chunk_size=500, overlap=100)
print('length of chunk',len(chunks))

Start: 0
End: 1000
chunk: SAYON BHATTACHARJEE 
Azure Data Engineer | Pune, India 
+91-8336004237 | sayon.bhattacharjee212@gmail.com  
 LinkedIn | GitHub | Medium   
 
SKILLS 
Azure Cloud Services – Azure Data Factory, Databricks, Data Flows, Synapse Analytics, Azure SQL database,  
Azure Data Lake, Key Vault, Azure Functions, Logic Apps, Cosmos DB, Azure Dev Ops  (basic working 
knowledge), MS Fabric (started exploring) 
Big Data Tools – Databricks, Apache Spark, Pyspark 
Datawarehouse and Databases – Azure SQL database, Azure Synapse Analytics, Delta Lake 
Programming – SQL, Python 
Reporting Tools – Power BI (basic working knowledge), Power App, Power Automate 
Primary Projects  – ETL, Dat awarehousing, Lakehouse, Medallion architecture projects, Batch processing , 
Spark Structured Streaming 
Project Domains – Financial Services, Manufacturing, Supply Chain Management 
Gen AI – Langchain, RAG, Vector Databases, Databricks Genie 
 
EXPERIENCE - 8 years 
Rockwell Automation (Senior Da

['SAYON BHATTACHARJEE \nAzure Data Engineer | Pune, India \n+91-8336004237 | sayon.bhattacharjee212@gmail.com  \n LinkedIn | GitHub | Medium   \n \nSKILLS \nAzure Cloud Services – Azure Data Factory, Databricks, Data Flows, Synapse Analytics, Azure SQL database,  \nAzure Data Lake, Key Vault, Azure Functions, Logic Apps, Cosmos DB, Azure Dev Ops  (basic working \nknowledge), MS Fabric (started exploring) \nBig Data Tools – Databricks, Apache Spark, Pyspark \nDatawarehouse and Databases – Azure SQL database, Azure Synapse Analytics, Delta Lake \nProgramming – SQL, Python \nReporting Tools – Power BI (basic working knowledge), Power App, Power Automate \nPrimary Projects  – ETL, Dat awarehousing, Lakehouse, Medallion architecture projects, Batch processing , \nSpark Structured Streaming \nProject Domains – Financial Services, Manufacturing, Supply Chain Management \nGen AI – Langchain, RAG, Vector Databases, Databricks Genie \n \nEXPERIENCE - 8 years \nRockwell Automation (Senior Data En

# Load Chunks into table

In [0]:
%sql
DROP TABLE IF EXISTS llm_dev.resume_bot;

CREATE TABLE IF NOT EXISTS llm_dev.resume_bot (
  row_id STRING,
  chunk STRING
);

In [0]:
for id, data in enumerate(chunks):
    spark.sql(r""" INSERT INTO llm_dev.resume_bot VALUES ('{}', '{}')""".format(id,data))

# PRE-REQUISITE ENDS HERE

# Section Below will go into the py file - I have tested here in notebook 

# Create Vectors & embeddings on the chunk from UI

In [0]:
def generate_embeddings(text):
    client = OpenAI(
                api_key="xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",
                base_url="https://dbc-xxxxxxxxxx-xxx.cloud.databricks.com/serving-endpoints"
                )
    response = client.embeddings.create(
                model='databricks-gte-large-en',
                input=text
                )
    return response

# RAG retriever

In [0]:
def rag_retriever(query_vector):
    from databricks.vector_search.client import VectorSearchClient
    vsc = VectorSearchClient()
    index = vsc.get_index(endpoint_name="best_practices_db", index_name="workspace.llm_dev.idx_resume_qa")
    results = index.similarity_search(query_vector=query_vector, columns="chunk", num_results=3)
    return results['result']['data_array']

# Instructions

In [0]:
name = "Sayon Bhattacharjee"

system_prompt = f"""You are acting as {name}. You are answering questions on {name}'s LinkedIn profile,
particularly questions related to {name}'s career, background, skills and experience.
Your responsibility is to represent {name} for interactions on the website as faithfully as possible.
Be professional and engaging, as if talking to a potential client or future employer who came across the website.
If you don't know the answer to any question tell you dont know clearly.
You are equipped with send_email tool to send mail to Sayon.
If a user requests to get in touch with a specific message then only you 
should use the send_email tool by creating a subject and use the user question as it is in email message body.
"""

# Send Email Tool

In [0]:
def send_email(subject: str, html_body: str):
    config = brevo_python.configuration.Configuration()
    config.api_key['api-key'] = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
    api_instance = brevo_python.TransactionalEmailsApi(brevo_python.ApiClient(config))

    send_smtp_mail = brevo_python.SendSmtpEmail(
        sender = {"name":"Sayon" , "email":"xxxx.bxxxxiems@gmail.com"},
        to = [{"email":"xxxx.bixxxxems@gmail.com" , "name": "Sayon"}] , 
        subject = subject,
        html_content = html_body
    )
    response = api_instance.send_transac_email(send_smtp_mail)

send_email_json = {
    "name" : "send_email",
    "description" : "Always use this tool to send email",
    "parameters": {
        "type": "object",
        "properties": {
            "subject": {
                "type": "string",
                "description": "The email subject"
            },
            "html_body": {
                "type": "string",
                "description": "The email body"
            },
        },
        "required": ["subject","html_body"],
        "additionalProperties": False
    }
}    

In [0]:
tools = [{"type": "function", "function": send_email_json}]

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        if tool_name == "send_email":
            result = send_email(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results


In [0]:
def chat(question):
    query_vector = generate_embeddings(question).data[0].embedding
    context = rag_retriever(query_vector)

    client = OpenAI(
                api_key="xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",
                base_url="https://xxx-xxxxx-xxx.cloud.databricks.com/serving-endpoints"
                )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Question: {question}\n\nContext from knowledge base:\n{context}"}
    ]

    response = client.chat.completions.create(
            messages=messages,
            model="databricks-meta-llama-3-3-70b-instruct",
            max_tokens=256, tools = tools
            )
    
    finish_reason = response.choices[0].finish_reason
    if finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
    else:
        done = True

    print(response.choices[0].message.content)
    print(response)

In [0]:
question = "Do you have experience in Azure cloud?"
chat(question)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Yes, I have experience in Azure cloud. Throughout my career, I have worked with various Azure services such as Azure Data Factory, Data Lake, Databricks, Azure SQL, ADLS, Cosmos DB, Azure Service Bus, Azure functions, and Delta tables. I have developed fully automated data pipelines using Azure Data Factory, Data Lake, and Databricks, and have also worked on projects that involved real-time customer product registrations using Databricks structured streaming job. Additionally, I have experience in optimizing slow-running Power BI reports by optimizing the underlying data. I

[Trace(trace_id=tr-df4457b9fb0e88a7c1f2a21bcce69a73), Trace(trace_id=tr-be9f999056c7a7ea662071ece2d30b3b)]